In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df['Delivery_Time'].hist()

In [ ]:
# Task 1: Write your code here:
df=df.drop(columns='Order_ID')

In [ ]:
# Task 2: Write your code here:


nums = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")
nums
cat= df.select_dtypes(include=["object"]).columns
for col in nums:
      df[col]=df[col].fillna(df[col].mean())
for col in cat:

    df[col]=df[col].fillna(df[col].mode()[0])

df.isnull().sum()
df["Delivery_Time"]=df["Delivery_Time"].fillna(df["Delivery_Time"].mean())

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Duplicates: {duplicates}")
df.drop_duplicates(inplace= True)
duplicates = df.duplicated().sum()
print(f"Duplicates: {duplicates}")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in cat:
   df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import  StandardScaler

scaler = StandardScaler()
df2=df.copy()
df2[nums] = scaler.fit_transform(df[nums])

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
X = df2.drop('Delivery_Time', axis=1).astype(float)
y = df2["Delivery_Time"].astype(float)
df2.isnull().sum()
y

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse
kf = KFold(n_splits=7, shuffle=True, random_state=42)
n_splits=5

scores=[]
for fold_idx, (train_index, test_index) in enumerate(kf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model= RandomForestRegressor(n_estimators=200)

  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Calculate metrics
  mse = sklearn_mse(y_test, y_pred)
  scores+=[mse]
  print(f"error: {fold_idx + 1}: ",mse,'\n')
print(f'the MSE avg across fold: {np.average(scores)}')

In [ ]:
# Task 1: Write your code here:
importances = model.feature_importances_
absolute_coef = np.abs(importances)
sorted_idx = np.argsort(absolute_coef)

plt.figure(figsize=(15, 6))
features = X.columns
plt.bar(features[sorted_idx], importances[sorted_idx],color='pink',edgecolor='purple')
plt.xlabel("Coefficient Value (Impact)")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code her:
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test, y_pred, alpha=0.5, s=10, c='pink')

# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual )', fontsize=12)
plt.ylabel('Predicted )', fontsize=12)
plt.title('Predicted vs Actual ', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:

from sklearn.ensemble import RandomForestClassifier

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error as mae

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),

  "CatBoost": CatBoostRegressor(verbose=0)
}

MAE = {}

for name in models:
  MAE[name] = []


kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  preds=[]
  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    preds+=[y_pred]
    # Calculate metrics
    m =mae(y_test,y_pred)

    # Store results
    MAE[model_name].append(m)
  print(f'The avg MAE<avg >for fold{fold_idx+1} is: ',np.average(preds))